In [3]:
from agents import OpenAIChatCompletionsModel, AsyncOpenAI
from dotenv import load_dotenv
import os

load_dotenv()


gemini_api_key = os.environ.get('GEMINI_API_KEY')
# OPENAI_API_KEY = os.environ.get('OPENAI_API_KEY')


if not gemini_api_key:
    raise ValueError('Gemini API KEY NOT FOUND')


external_client = AsyncOpenAI(
    api_key=gemini_api_key,
    base_url='https://generativelanguage.googleapis.com/v1beta/openai'
    # api_key=OPENAI_API_KEY,
    # base_url='https://api.openai.com/v1/chat/completions'
)


model = OpenAIChatCompletionsModel(
    # model = 'gpt-3.5-turbo',
    # model = 'chatgpt-4o-latest',
    model='gemini-2.0-flash',
    openai_client=external_client
)

### Input GuardRail

In [4]:
from pydantic import BaseModel
from agents import Agent, Runner, InputGuardrailTripwireTriggered, input_guardrail, RunContextWrapper, GuardrailFunctionOutput


class ProgrammingTopicsOutput(BaseModel):
    is_programming_topic: bool
    reasoning: str


guardrail_agent = Agent(
    name = "Input Guardrail Agent",
    instructions = "Check if the user asking for programming related questions or not",
    output_type=ProgrammingTopicsOutput
)


@input_guardrail
async def programming_guardrail(ctx: RunContextWrapper[None],agent,input)->GuardrailFunctionOutput:
    result = await Runner.run(guardrail_agent, input, context=ctx)

    return GuardrailFunctionOutput(
        output_info=result.final_output,
        tripwire_triggered=result.final_output.is_programming_topic
    )


agent = Agent(
    name = "Test Agent",
    instructions="You are programming agent which takes programming test",
    input_guardrails=[programming_guardrail]
)


try:
    await Runner.run(agent,"Can you give me the Roadmap of learning MERN Stack")
    print('Successfully Run! No Guardrail Tripped')
except InputGuardrailTripwireTriggered:
    print('Guardrail Tripped! Asking about programming topic')


Error getting response: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}. (request_id: req_0f605a0d3ba7465095aaf5b6453633c0)


RateLimitError: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}

Error getting response: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}. (request_id: req_c7d1dc8cc4774e38941f5332db04d493)


### Output GuardRail

In [ ]:
from pydantic import BaseModel
from agents import Agent, output_guardrail, RunContextWrapper, OutputGuardrailTripwireTriggered

class ProgrammingOutput(BaseModel):
    reasoning: str
    is_programming: bool

class MessageOutput(BaseModel):
    response: str


guardrail_agent = Agent(
    name = "Guardrail Agent",
    instructions = "Check if the user includes any programming or software related topic",
    output_type = [ProgrammingOutput]
)


@output_guardrail
async def programming_output_guardrail(ctx:RunContextWrapper[None], agent, output):
    result = await Runner.run(agent, output.response, context=ctx)

    return GuardrailFunctionOutput(
        output_info=result.final_output,
        tripwire_triggered=result.final_output.is_programming
    )


agent = Agent(
    name="Virtual Assistant",
    instructions="You are a company virtual assistant",
    output_guardrails=[programming_output_guardrail],
    output_type=[MessageOutput]
)


try:
    await Runner.run(agent, 'I am new to technology, which career I choose AI or Web Development')
    print("Success! GuardRail Didnt trip")
except OutputGuardrailTripwireTriggered:
    print()